# ASTR 457: The Midterm

Posted Thu Oct 8, due Wed Oct 14 by Noon (fork → PR, as always). Graded like a lab.

Three parts. Each is designed to test something every lab so far has come back to - which error bar is the right one, whether the model needs the extra parameter, and whether the check you ran could have failed. You've now got the tools to do all of this, so let's do it on numbers you haven't seen: 250 real Type Ia supernovae, and someone else's analysis of real white dwarfs.

Part 1 is 30 pts (6 each), Part 2 is 45 pts, and Part 3 is 25 pts. You'll need 2b and 2c right to get 2d right.

Remember, the rules are the syllabus rules - open book, you can use AI tools (document them in 3b), you can work together and discuss problems and solutions, but you have to write up your own solution and run it on **your own data** in `data/<your netid>/` (nobody else's numbers will match yours).

Put your notebook in `submissions/<your netid>/midterm.ipynb`. The path below (`DATA`) reaches your files from there. Make sure it runs top to bottom from a fresh kernel ("Restart & Run All") - that is how it will be graded, and how it will be run if you defend it.

Throughout, make sure every number you quote has a unit, an error bar and a sensible number of digits (two significant figures on the error bar, and round the value to match).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize

NETID = 'anuvacg2'   # <-- set this
# DATA = f'data/{NETID}/' # direct in folder 
DATA = f'../../labs/midterm/data/{NETID}/'   # from submissions/<your netid>/

def load(name):
    """Read one of your CSV files as a pandas table; d['column'].to_numpy() gives a numpy array. Names stay strings."""
    return pd.read_csv(DATA + name + '.csv', dtype={'name': str})

## Part 1: Error bars (30 pts, 6 each)

Five short scenarios. Each one is a number someone handed you - say whether it's right, compute the right one, and say in a sentence or two why. Show the code. Each of these should be about 20 minutes of work.

### 1a - The extinction coefficient (`p1a_extinction.csv`)

An observer measured a photometric zero point at 15 different airmasses through one night. Extinction in magnitudes grows in proportion to the airmass $X$, so the zero point should follow a line, $m = k\,X + m_0$, where $k$ is the extinction coefficient in mag per airmass. Columns: `airmass`, `zeropoint_mag`, `sigma_quoted_mag` (the pipeline's error on each point).

1. Fit the line by weighted least squares with the quoted errors. Report $k$ and its error bar computed from the quoted errors alone.
2. Now run `np.polyfit(airmass, zeropoint_mag, 1, w=1/sigma_quoted_mag, cov=True)` and report its error bar on $k$. Explain exactly what `polyfit` did to get a different number (read its documentation for `cov`).
3. Compute $\chi^2$ and the number of degrees of freedom. How probable is a $\chi^2$ this large or this small if the quoted errors were right? (Compute it from the $\chi^2$ distribution.)
4. Report the error bar on $k$ you'd quote, and justify it. What does your answer to (3) tell you about the pipeline's error bars?

In [ ]:
# Helper Functions: 
def fmt(v, e, unit=''):
    """value ± error, error to 2 s.f., value rounded to match."""
    d = 1 - int(np.floor(np.log10(abs(e))))
    return f"{round(v, d):.{max(d, 0)}f} ± {round(e, d):.{max(d, 0)}f} {unit}".strip()

def fmt_asym(v, lo, hi, unit=''):
    d = 1 - int(np.floor(np.log10(min(lo, hi))))
    f = lambda x: f"{round(x, d):.{max(d, 0)}f}"
    return f"{f(v)} +{f(hi)} / -{f(lo)} {unit}".strip()

In [ ]:
# 1a: your work here
d1a = load('p1a_extinction')
X, zp, sig = d1a['airmass'].to_numpy(), d1a['zeropoint_mag'].to_numpy(), d1a['sigma_quoted_mag'].to_numpy()
N = len(X); dof = N - 2

# (1) weighted least squares by hand (Lab 04): cov = (A^T W A)^-1, theta = cov A^T W y
A = np.column_stack([X, np.ones(N)])
W = 1 / sig**2
cov = np.linalg.inv(A.T @ (A * W[:, None]))
k, m0 = cov @ (A.T @ (W * zp))
sig_k, sig_m0 = np.sqrt(np.diag(cov))
print(f"(1) k  = {fmt(k, sig_k, 'mag/airmass')}   (quoted errors only)")
print(f"    m0 = {fmt(m0, sig_m0, 'mag')}")

# (2) np.polyfit with default cov=True
p_pf, C_pf = np.polyfit(X, zp, 1, w=1/sig, cov=True)
sig_k_pf = np.sqrt(C_pf[0, 0])
print(f"(2) polyfit: k = {p_pf[0]:.4f}, sigma_k = {sig_k_pf:.4f} mag/airmass")

# (3) chi^2 and its tail probabilities
chi2 = np.sum(((zp - A @ [k, m0]) / sig)**2)
p_big, p_small = stats.chi2.sf(chi2, dof), stats.chi2.cdf(chi2, dof)
print(f"(3) chi2 = {chi2:.2f} for {dof} dof (expect {dof} ± {np.sqrt(2*dof):.1f});  reduced = {chi2/dof:.2f}")
print(f"    P(chi2 >= obs) = {p_big:.2e},  P(chi2 <= obs) = {p_small:.2e}")
print(f"    sigma_polyfit / sigma_quoted = {sig_k_pf/sig_k:.3f};  sqrt(chi2/dof) = {np.sqrt(chi2/dof):.3f}")

# (4) rescale by sqrt(chi2_red); sanity-check the residuals first
sig_k_final = sig_k * np.sqrt(chi2 / dof)
print(f"(4) k = {fmt(k, sig_k_final, 'mag/airmass')}")

plt.errorbar(X, (zp - A @ [k, m0]) / sig, yerr=1, fmt='o', capsize=3)
plt.axhline(0, color='k', lw=0.8); plt.xlabel('airmass'); plt.ylabel('(data - fit) / sigma_quoted')
plt.title('Standardized residuals'); plt.show()

**ANSWER:**

1. k = 0.149 $\pm$ 0.011 mag/airmass
2. With polyfit, k = 0.1489 $\pm$ 0.0243 mag/airmass. This is because this considers the relative weights of the scatter adjusts by the red chi_square value. By default `(cov=True)` it treats w as only relative weights and multiplies the covariance by $\chi^2/{\rm dof}$. However, using `cov='unscaled'` would have the intended outcome. 
3. The chi_squared is 63.74 for 13 dof. This value is significantly too large if the quoted errors were correct. The probability of a $\chi^2$ this large or larger is $1.1\times10^{-8}$, so the quoted errors cannot be accurate.
4. My final value is k = 0.1489 $\pm$ 0.0243 mag/airmass. The pipeline's error bars here are too small by about a factor of sqrt(4.9). The residuals show no obvious trend with airmass, so the line is fine. Rescaling assumes a roughly uniform underestimate of the errors.

### 1b - The centroid (`p1b_centroid.csv`)

A classmate measured the position of a star from `n_photons` photons, each landing at a position drawn from a Gaussian of width `psf_sigma_arcsec` around the true position. They report an error on the position of `quoted_sigma_x0_arcsec`, from a clever weighting scheme they found online.

1. Compute the Cramér-Rao bound on the error of any unbiased estimator of the position (Lab 02).
2. Is the quoted error possible? If not, say what the classmate's number must mean instead, and name two concrete mistakes that would produce an error bar this small.

In [ ]:
# 1b: your work here

d1b = load('p1b_centroid')
n_ph  = d1b['n_photons'].iloc[0]
s_psf = d1b['psf_sigma_arcsec'].iloc[0]
s_q   = d1b['quoted_sigma_x0_arcsec'].iloc[0]

# Gaussian with known width: d lnp/dx0 = (x-x0)/s^2  ->  I_1 = 1/s^2  ->  I = N/s^2
crb = s_psf / np.sqrt(n_ph)
print(f"N = {n_ph},  PSF sigma = {s_psf} arcsec")
print(f"Cramer-Rao bound: {crb:.4g} arcsec;  quoted: {s_q:.4g} arcsec;  quoted/CRB = {s_q/crb:.3g}")
print(f"quoted = sigma * N^-q with q = {-np.log(s_q/s_psf)/np.log(n_ph):.3f}   (CRB has q = 0.5)")

cands = {'sigma/N': s_psf/n_ph, 'sigma^2/N (variance, wrong units)': s_psf**2/n_ph,
         'sigma/sqrt(N)/2.355 (FWHM->sigma applied twice)': crb/2.3548, 'sigma/sqrt(2N)': s_psf/np.sqrt(2*n_ph)}
for name, v in cands.items():
    print(f"  candidate {name:50s} = {v:.4g}   (quoted/candidate = {s_q/v:.3g})")

**ANSWER:**

The Cramer-Rao bound is 0.1025 arcsec. The classmate's quoted error of 0.07913 arcsec cannot be accurate for an unbiased estimator, since there are only 233 independent photons. For a bound this low, you would need almost 400. It does not match $\sigma/N$ or the variance either, so it is most likely the self-consistency scatter of a biased estimator. This error could be from counting certain photons multiple times or treating correlated photons as independent ones. 

### 1c - The transit depth's error bar (`p1c_profile.csv`)

A transit-depth fit produced this profile: columns `depth_ppt` (the depth, in parts per thousand) and `minus2lnL` ($-2\ln L$ at that depth, other parameters fixed at their best values). A classmate took the second derivative of `minus2lnL` at its minimum and reported $\sigma = 1/\sqrt{d^2(-2\ln L)/d\,\textrm{depth}^2}$.

1. Find the best depth and the correct $1\sigma$ error from the curvature. Is the classmate's error bar right? If not, by what factor is it off, and why?
2. Read the $\Delta(-2\ln L) = 1$ interval directly off the profile. Is it symmetric? Compare it with your curvature answer.
3. If the classmate quotes $\pm$ their $\sigma$ on many independent datasets, what fraction of those intervals contain the true depth? Predict it, then check it with a simulation (a few thousand fake best-fit depths drawn around a truth you choose, using your correct $\sigma$).

In [ ]:
# 1c: your work here

from scipy.interpolate import CubicSpline
from scipy.optimize import minimize_scalar, brentq

d1c = load('p1c_profile').sort_values('depth_ppt')
dep, m2 = d1c['depth_ppt'].to_numpy(), d1c['minus2lnL'].to_numpy()
cs = CubicSpline(dep, m2)

# 1. best depth and curvature 
i0 = np.argmin(m2)
r = minimize_scalar(lambda x: float(cs(x)), bounds=(dep[max(i0-1, 0)], dep[min(i0+1, len(dep)-1)]), method='bounded')
d0, m2min = r.x, r.fun
curv = float(cs(d0, 2))
sig_curv = np.sqrt(2 / curv)       # -2lnL ~ (d-d0)^2/sigma^2  =>  curv = 2/sigma^2
sig_cm   = 1 / np.sqrt(curv)       # the classmate's formula
sel = (m2 - m2min) < 2             # cross-check: local quadratic fit
if sel.sum() >= 5:
    print(f"quadratic-fit curvature = {2*np.polyfit(dep[sel], m2[sel], 2)[0]:.4g}  vs spline {curv:.4g} (ppt^-2)")
print(f"best depth = {d0:.4f} ppt;  d2(-2lnL)/dd^2 = {curv:.4g} ppt^-2")
print(f"correct sigma = {sig_curv:.4g} ppt;  classmate sigma = {sig_cm:.4g} ppt;  ratio = {sig_curv/sig_cm:.4f}")

# 2. Delta(-2lnL) = 1 interval, read off the profile (profile must exceed +1 at both ends)
f = lambda x: float(cs(x)) - m2min - 1
lo, hi = brentq(f, dep[0], d0), brentq(f, d0, dep[-1])
print(f"Delta=1 interval: [{lo:.4f}, {hi:.4f}] ppt  ->  depth = {fmt_asym(d0, d0-lo, hi-d0, 'ppt')}")
print(f"  minus side = {d0-lo:.4g}, plus side = {hi-d0:.4g};  curvature sigma = {sig_curv:.4g}  (all ppt)")

xx = np.linspace(dep[0], dep[-1], 400)
plt.plot(dep, m2 - m2min, 'ko', ms=3, label='profile')
plt.plot(xx, (xx - d0)**2 / sig_curv**2, 'C0-', label=r'parabola with $\sigma=\sqrt{2/f\,\prime\prime}$')
plt.axhline(1, color='0.5', ls=':')
for v in (lo, hi): plt.axvline(v, color='C2', ls='--')
for v in (d0 - sig_cm, d0 + sig_cm): plt.axvline(v, color='C3', ls=':')
plt.plot([], [], 'C2--', label=r'$\Delta=1$ interval'); plt.plot([], [], 'C3:', label=r"classmate's $\pm\sigma$")
plt.ylim(-0.5, 4); plt.xlabel('depth [ppt]'); plt.ylabel(r'$\Delta(-2\ln L)$'); plt.legend(); plt.show()

# 3. coverage: prediction and simulation
rng = np.random.default_rng(457)
truth = d0; dhat = rng.normal(truth, sig_curv, 20000)
for lab, s in [("classmate's sigma", sig_cm), ('correct sigma', sig_curv)]:
    c = np.mean(np.abs(dhat - truth) < s)
    pred = 2 * stats.norm.cdf(s / sig_curv) - 1
    print(f"{lab:18s}: predicted {pred:.3f}, simulated {c:.3f} ± {np.sqrt(c*(1-c)/len(dhat)):.3f}")

**ANSWER:**

1. The best depth is 2.4164 ppt with a sigma of 0.16 ppt. The classmates reported sigma of 0.1131 is a factor of sqrt(2) = 1.4142 off from my value, which makes it seem like they just dropped the factor of 2 in their calculation. 
2. The $\Delta(-2\ln L) = 1$ interval is [2.2561, 2.5755], which is symmetric around the best-depth calculation and agrees with the curvature sigma. This works because the profile is already very nearly parabolic.
3. An interval of $\pm\sigma/\sqrt2=\pm0.707\sigma$ has a predicted coverage of 52.0% instead of 68.3%. This aligns with the simulated values, where the classmate's sigma ended up as 0.523 $\pm$ 0.004 and the correct sigma was 0.679 $\pm$ 0.003.  

### 1d - The companion mass (`p1d_mass_samples.csv`)

These are 8000 posterior samples of the mass of an unseen companion, `M2_Msun`. Three classmates report its uncertainty as (i) the variance of the samples, (ii) the 84th percentile minus the 16th, and (iii) the standard deviation divided by $\sqrt{8000}$.

1. Report the mass as a median with its 16th-84th percentile error bars, in solar masses.
2. For each of (i)-(iii): what quantity is it actually, what are its units, and by roughly what factor is it wrong as a $1\sigma$ error bar?
3. Is a symmetric $\pm$ error bar appropriate for this posterior? If not, show why.

In [ ]:
# 1d: your work here

d1d = load('p1d_mass_samples')
M = d1d['M2_Msun'].to_numpy(); n = len(M)

# 1. median with 16th-84th percentile error bars
p16, p50, p84 = np.percentile(M, [16, 50, 84])
print(f"M2 = {fmt_asym(p50, p50-p16, p84-p50, 'Msun')}")

#  2. the three classmates' numbers
ref = 0.5 * (p84 - p16) 
var, w68, sem = M.var(ddof=1), p84 - p16, M.std(ddof=1) / np.sqrt(n)
print(f"(i)   variance      = {var:.4g} Msun^2   (numerically {var/ref:.3g} x the half-width; units wrong)")
print(f"(ii)  p84 - p16     = {w68:.4g} Msun   ({w68/ref:.3g} x the half-width)")
print(f"(iii) std/sqrt(8000)= {sem:.4g} Msun   ({sem/ref:.3g} x the half-width; 1/sqrt(n) = {1/np.sqrt(n):.4f})")

# 3. is a symmetric +- appropriate?
mu, sd = M.mean(), M.std(ddof=1)
print(f"mean = {mu:.4g}, median = {p50:.4g}, std = {sd:.4g} Msun, skew = {stats.skew(M):.2f}")
print(f"minus side = {p50-p16:.4g}, plus side = {p84-p50:.4g} Msun  (plus/minus = {(p84-p50)/(p50-p16):.2f})")
print(f"{'pct':>6} {'actual':>10} {'Gaussian':>10}")
for q in [2.5, 16, 50, 84, 97.5]:
    print(f"{q:6.1f} {np.percentile(M, q):10.4g} {stats.norm.ppf(q/100, mu, sd):10.4g}")

plt.hist(M, bins=80, density=True, color='0.8', edgecolor='k', lw=0.2)
g = np.linspace(M.min(), M.max(), 400); plt.plot(g, stats.norm.pdf(g, mu, sd), 'C1', label='Gaussian, same mean/std')
for v in (p16, p50, p84): plt.axvline(v, color='C0', ls='--')
for v in (mu - sd, mu + sd): plt.axvline(v, color='C1', ls=':')
plt.xlabel(r'$M_2$ [$M_\odot$]'); plt.ylabel('posterior density'); plt.legend(); plt.show()

**ANSWER:**

1. M2 = 6.9 +2.1 / -1.6 Msun
2. The variance is reported as 3.7 M_Sun^2, which has the wrong units on the error bar. It's value is also equal to 2.02 times the half-width, which is a coincidence of the unit choice. The 68% interval width is correctly reported at 3.7 M_Sun, although the error bar itself should be half of that value and not the full interval. The standard deviation adjusted by 8000 samples (std / sqrt(8000)) is reported as 0.02159, which is significantly too small since it is the Monte Carlo standard error on the mean of the samples, not the posterior width. 
3. From the plot generated above, we can see that the curve is almost Gaussian but has a small right-skew with a longer right tail. Because of this, a symmetric error bar is not actually appropriate, since it is a poor approximation but not the strongest estimator as it misplaces the median. I would quote the asymmetric 16–84 interval around the median instead. 

### 1e - The slope at fixed intercept (`p1e_slope_intercept_samples.csv`)

Posterior samples of a slope and an intercept from a line fit. A classmate fixed the intercept at its best value, refit only the slope, and quoted the slope's error from that 1-D fit.

1. Compute the slope's error bar the classmate got (the spread of the slope *at fixed intercept*), and the marginal one (the spread of the slope with the intercept marginalized over).
2. Compute their ratio, and the correlation coefficient $\rho$ of the samples. How does the ratio depend on $\rho$? Plot the samples and draw both error bars on the plot.
3. When is the classmate's number the right one to quote? One sentence.

In [ ]:
# 1e: your work here
d1e = load('p1e_slope_intercept_samples')
print(d1e.columns.tolist()) 
cm = next((c for c in d1e.columns if 'slope' in c.lower()), d1e.columns[0])
cb = next((c for c in d1e.columns if 'inter' in c.lower()), d1e.columns[1])
m, b = d1e[cm].to_numpy(), d1e[cb].to_numpy()
um = '(y-unit)/(x-unit)' 

# 1. marginal vs fixed-intercept error on the slope
S = np.cov(m, b)
rho = S[0, 1] / np.sqrt(S[0, 0] * S[1, 1])
sig_marg = np.sqrt(S[0, 0])
sig_fix  = 1 / np.sqrt(np.linalg.inv(S)[0, 0]) 

# empirical check: spread of the slope in a thin slab of intercept around its best value
b0, m0 = np.median(b), np.median(m)
slab = np.abs(b - b0) < 0.1 * b.std(ddof=1)
print(f"marginal  sigma_m = {sig_marg:.4g} {um}")
print(f"fixed-b   sigma_m = {sig_fix:.4g} {um}   (thin-slab check: {m[slab].std(ddof=1):.4g}, n = {slab.sum()})")

# 2. ratio and correlation
print(f"rho = {rho:.3f};  ratio marginal/fixed = {sig_marg/sig_fix:.3f};  1/sqrt(1-rho^2) = {1/np.sqrt(1-rho**2):.3f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
ax[0].plot(m, b, '.', ms=2, alpha=0.25, color='0.5')
ax[0].axhline(b0, color='k', lw=0.6, ls=':')
ax[0].errorbar(m0, b0, xerr=sig_fix, fmt='o', color='C3', capsize=5, lw=3, label=f'fixed intercept: ±{sig_fix:.3g}')
ax[0].errorbar(m0, b0 - 2.2 * b.std(), xerr=sig_marg, fmt='o', color='C0', capsize=5, lw=3, label=f'marginalized: ±{sig_marg:.3g}')
for v in (m0 - sig_marg, m0 + sig_marg): ax[0].axvline(v, color='C0', ls='--', lw=0.8)
ax[0].set_xlabel(f'slope [{um}]'); ax[0].set_ylabel('intercept [y-unit]'); ax[0].legend(fontsize=8)

r_ = np.linspace(0, 0.99, 200)
ax[1].plot(r_, 1 / np.sqrt(1 - r_**2), 'k'); ax[1].plot(abs(rho), sig_marg / sig_fix, 'C3o', label='your samples')
ax[1].set_xlabel(r'$|\rho|$'); ax[1].set_ylabel('marginal / fixed-intercept error'); ax[1].legend()
plt.tight_layout(); plt.show()

**ANSWER:**

1. Fixed intercept: sigma_m = 0.0135 (y-unit)/(x-unit) \
Marginal slope: sigma_m = 0.03683 (y-unit)/(x-unit) 
2. The correlation is $\rho = -0.930$ and the ratio is $0.0368/0.0135 = 2.73$ The ratio is 1 at $\rho = 0$ and diverges as $|\rho|\to1$. This makes sense since the posterior is a narrow tilted ellipse. The fixed-intercept only slices across a single part of the narrow width, while the marginalized one projects more of the whole ellipse, including its long axis, onto the slope axis.
3. The classmates' number is the right one to quote only when the intercept is known independently or when $\rho = 0$. 

## Part 2: 250 supernovae and a step (45 pts)

Type Ia supernovae are standardizable candles: their peak brightness, corrected for how fast they fade and for their color, measures distance well enough to have discovered cosmic acceleration. Your file `p2_pantheon_sne.csv` holds 250 real SNe Ia from the Pantheon+ compilation (Scolnic et al. 2022, ApJ 938, 113; [Brout et al. 2022](https://arxiv.org/abs/2202.04077), ApJ 938, 110), all at redshift $0.023 < z < 0.15$. For each supernova, a light-curve fit (SALT2) gives:

- `mB`: the peak apparent magnitude in the B band, and `mB_err`;
- `x1`: the light-curve *stretch* (broader light curves are brighter), and `x1_err`;
- `c`: the *color* (redder supernovae are fainter, from dust and intrinsically), and `c_err`;
- `cov_mB_x1`, `cov_mB_c`, `cov_x1_c`: the covariances between those three, from the same light-curve fit;
- `z`: the redshift; `host_logmass`, `host_logmass_err`: $\log_{10}$ of the host galaxy's stellar mass in solar masses, and its error.

The standardized brightness is the **Tripp relation**,

$$m_B = M + \mu(z) - \alpha\,x_1 + \beta\,c + \gamma\,\Theta(\log M_\star - 10),$$

where $M$ is the absolute magnitude of a "standard" SN Ia, $\mu(z)$ the distance modulus, $\Theta$ is 1 for hosts more massive than $10^{10}\,M_\odot$ and 0 otherwise, and $\gamma$ is the **host-mass step**. Supernovae in massive hosts appear to be brighter after standardization (Kelly et al. 2010; Sullivan et al. 2010); whether the step is real, and why, matters for supernova dark-energy measurements. In this form, a step that makes supernovae in massive hosts brighter has $\gamma < 0$; the published value is about $\gamma \approx -0.06$ mag. So the question for your sample is whether you need $\gamma$ at all.

There are two more things you'll need. At these redshifts the galaxies' own motions (peculiar velocities, about 250 km/s) shift $z$, and so $\mu$; add $\sigma_{\mu,\rm pec} = \frac{5}{\ln 10}\frac{250\ {\rm km/s}}{c\,z}$ to each supernova's variance. And no supernova is a perfect standard candle: add an intrinsic scatter $\sigma_{\rm int}$, a free parameter. Use a flat $\Lambda$CDM distance modulus with $H_0 = 70$ km/s/Mpc and $\Omega_m = 0.3$ (`astropy.cosmology.FlatLambdaCDM(H0=70, Om0=0.3).distmod(z)`); $M$ absorbs the choice of $H_0$.

We planted between **two and four contaminants** in your file - objects whose `mB` we made fainter by more than a magnitude, so they are not standard SNe Ia any more (think of a mis-classified peculiar supernova). Their names are not in the file. Real data have a few real outliers too, so not everything far from the line is planted.

### 2a - Look at the data (5 pts)

1. Compute $\mu(z)$ and plot the **Hubble residuals** $m_B - \mu(z)$ against $x_1$, against $c$, and against `host_logmass`.
2. Which way do the $x_1$ and $c$ trends go? Is that the sign the Tripp relation expects? Two or three sentences.
3. The raw trend with host mass may not have the sign of $\gamma$. Why not? (Plot $x_1$ against host mass.)
4. Can you see any of the contaminants?
5. What units do $\alpha$ and $\beta$ have?

In [ ]:
# 2a: your work here

from astropy.cosmology import FlatLambdaCDM

sn = load('p2_pantheon_sne')
z, mB = sn['z'].to_numpy(), sn['mB'].to_numpy()
x1, col = sn['x1'].to_numpy(), sn['c'].to_numpy()      # 'col', since c is also the speed of light
logM = sn['host_logmass'].to_numpy()
mu = FlatLambdaCDM(H0=70, Om0=0.3).distmod(z).value    # mag
resid = mB - mu                                        # Hubble residual = M - a*x1 + b*c + g*Theta + scatter
Theta = (logM > 10).astype(float)
print(f"{len(sn)} SNe, z = {z.min():.3f}-{z.max():.3f}; {int(Theta.sum())} in massive hosts (log M > 10)")

# crude OLS, outliers included: only to start the fits later and to spot the contaminants
Aols = np.column_stack([np.ones_like(z), x1, col, Theta])
coef = np.linalg.lstsq(Aols, resid, rcond=None)[0]
res_ols = resid - Aols @ coef
rsig = 1.4826 * np.median(np.abs(res_ols - np.median(res_ols)))
faint = np.where(res_ols > 4 * rsig)[0]; bright = np.where(res_ols < -4 * rsig)[0]
print(f"crude OLS: M={coef[0]:.3f}, alpha={-coef[1]:.3f}, beta={coef[2]:.3f}, gamma={coef[3]:.3f}; robust scatter={rsig:.3f} mag")
print("fainter than +4 robust sigma:", faint, np.round(res_ols[faint], 2), "| brighter than -4:", bright)

fig, ax = plt.subplots(2, 2, figsize=(11, 8.5))
for a, xv, lab in [(ax[0, 0], x1, r'$x_1$'), (ax[0, 1], col, r'$c$'),
                   (ax[1, 0], logM, r'host $\log_{10}(M_\star/M_\odot)$')]:
    a.plot(xv, resid, 'k.', ms=4)
    a.plot(xv[faint], resid[faint], 'ro', mfc='none', ms=10)
    lr = stats.linregress(xv, resid); xs = np.array([xv.min(), xv.max()])
    a.plot(xs, lr.intercept + lr.slope * xs, 'C0-')
    a.set_xlabel(lab); a.set_ylabel(r'$m_B-\mu(z)$ [mag]')
    a.set_title(f'raw slope = {lr.slope:.3g} ± {lr.stderr:.2g}', fontsize=11)
    print(f"Hubble residual vs {lab}: slope = {lr.slope:.3g} ± {lr.stderr:.2g}")
ax[1, 0].axvline(10, color='0.5', ls=':')
lr = stats.linregress(logM, x1)
ax[1, 1].plot(logM, x1, 'k.', ms=4); xs = np.array([logM.min(), logM.max()])
ax[1, 1].plot(xs, lr.intercept + lr.slope * xs, 'C0-'); ax[1, 1].axvline(10, color='0.5', ls=':')
ax[1, 1].set_xlabel(r'host $\log_{10}(M_\star/M_\odot)$'); ax[1, 1].set_ylabel(r'$x_1$')
ax[1, 1].set_title(f'x1 vs host mass: slope = {lr.slope:.3g} ± {lr.stderr:.2g} per dex', fontsize=11)
plt.tight_layout(); plt.show()

**ANSWER:**

The $x_1$ trend is negative with a slope of -0.16, and the $c$ trend is positive with a slope of 2.76. Both are signs that the Tripp relation expects since the coefficient term of $x_1$ is negative $\alpha$ and the coefficient term of $c$ is positive $\beta$. \
The raw trend with the host mass has a positive slope of 0.0964 ± 0.026, which is opposite from the given information that "a step that makes supernovae in massive hosts brighter has $\gamma < 0$ " because because host mass is correlated with stretch. Only the joint fit can separate $x_1$, $c$ and $\Theta$ since massive hosts have lower $x_1$. 
You can visually see the same three or four contaminants in the first 3 panels at $+1.04$, $+0.52$, $+1.26$ and $+1.44$ mag. The three largest are very likely the contaminants, but the +0.52 point is less extreme and may be an ordinary tail point. \
$\alpha$ has units of magnitude, and $\beta$ is dimensionless. 

### 2b - The likelihood (10 pts)

The parameters are $\theta = (M, \alpha, \beta, \gamma, \sigma_{\rm int})$. Each supernova's variance is

$$\sigma_i^2 = \mathbf{v}^T C_i\,\mathbf{v} + \sigma_{\mu,{\rm pec},i}^2 + \sigma_{\rm int}^2, \qquad \mathbf{v} = (1, \alpha, -\beta),$$

where $C_i$ is that supernova's $3\times3$ covariance of $(m_B, x_1, c)$.

1. Write out $\mathbf{v}^T C_i\,\mathbf{v}$ term by term, and explain in one sentence why the off-diagonal terms are there.
2. Write the log-likelihood. Remember the $\ln\sigma_i^2$ term - the variance depends on the parameters.
3. State a prior for every parameter, and justify each in one line.
4. Decide whether to sample $\ln\sigma_{\rm int}$ or $\sigma_{\rm int}$, and say why.
5. Decide what to do about the contaminants. You may model them or remove them, but say how you found them and what a cut would cost you.

In [ ]:
# 2b: your work here

C_LIGHT = 299792.458  # km/s
pec2 = (5 / np.log(10) * 250.0 / (C_LIGHT * z))**2 

D_all = dict(y=resid, x1=x1, col=col, Th=Theta,
             vmB=sn['mB_err'].to_numpy()**2, vx1=sn['x1_err'].to_numpy()**2, vc=sn['c_err'].to_numpy()**2,
             cmx=sn['cov_mB_x1'].to_numpy(), cmc=sn['cov_mB_c'].to_numpy(), cxc=sn['cov_x1_c'].to_numpy(),
             pec2=pec2)
names5 = ['M', 'alpha', 'beta', 'gamma', 'ln_sigma_int'] 

def sigma2(th, D):
    """v^T C v + sigma_pec^2 + sigma_int^2,  v = (1, alpha, -beta)"""
    M, a, b, g, lns = th
    return (D['vmB'] + a**2 * D['vx1'] + b**2 * D['vc']
            + 2*a*D['cmx'] - 2*b*D['cmc'] - 2*a*b*D['cxc']
            + D['pec2'] + np.exp(2 * lns))

def loglike(th, D):
    M, a, b, g, lns = th
    r = D['y'] - M + a*D['x1'] - b*D['col'] - g*D['Th']
    s2 = sigma2(th, D)
    return -0.5 * np.sum(r**2 / s2 + np.log(2 * np.pi * s2))

#  priors (normalized, so evidences are meaningful) 
GSD = 0.1 
LN_VOL = np.log(4.) + np.log(2.) + np.log(8.) + np.log(np.log(1e3)) 
def lp_core(M, a, b, lns):
    ok = (-21 < M < -17) and (-1 < a < 1) and (0 < b < 8) and (np.log(1e-3) < lns < 0)
    return -LN_VOL if ok else -np.inf
def logpost1(th, D, gsd=GSD): 
    M, a, b, g, lns = th
    lp = lp_core(M, a, b, lns)
    if not np.isfinite(lp): return -np.inf
    return lp - 0.5 * (g / gsd)**2 - np.log(gsd * np.sqrt(2 * np.pi)) + loglike(th, D)
def logpost0(t4, D): 
    M, a, b, lns = t4
    lp = lp_core(M, a, b, lns)
    return -np.inf if not np.isfinite(lp) else lp + loglike([M, a, b, 0.0, lns], D)

#  optimizer + finite-difference Hessian (Lab 02/03) 
H5 = np.array([1e-3, 1e-3, 1e-2, 2e-3, 5e-3]) 
def num_hess(f, x, h):
    n = len(x); E = np.diag(h); H = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            H[i, j] = (f(x+E[i]+E[j]) - f(x+E[i]-E[j]) - f(x-E[i]+E[j]) + f(x-E[i]-E[j])) / (4*h[i]*h[j])
    return H

def maximize(logf, th0, h):
    """Maximize logf. Returns (theta_hat, logf_max, covariance = inverse Hessian of -logf)."""
    def nlf(th):
        v = logf(th); return -v if np.isfinite(v) else 1e30
    opt = dict(xatol=1e-9, fatol=1e-10, maxiter=20000, maxfev=20000, adaptive=True)
    x = np.array(th0, float)
    for _ in range(3): x = minimize(nlf, x, method='Nelder-Mead', options=opt).x 
    return x, -nlf(x), np.linalg.inv(num_hess(nlf, x, h))

def start(D):
    A = np.column_stack([np.ones_like(D['y']), D['x1'], D['col'], D['Th']])
    cf = np.linalg.lstsq(A, D['y'], rcond=None)[0]
    return np.array([cf[0], -cf[1], cf[2], cf[3], np.log(max(0.5 * np.std(D['y'] - A @ cf), 0.02))])

#  contaminants
def sub(D, keep): return {k: v[keep] for k, v in D.items()}
def std_resid(th, D):
    M, a, b, g, lns = th
    return (D['y'] - M + a*D['x1'] - b*D['col'] - g*D['Th']) / np.sqrt(sigma2(th, D))

def clip_fit(D, K=4.0, max_iter=20):
    keep = np.ones(len(D['y']), bool)
    for _ in range(max_iter):
        Dk = sub(D, keep)
        th, _, _ = maximize(lambda t: logpost1(t, Dk), start(Dk), H5)
        zr = std_resid(th, D) 
        new_keep = zr < K 
        if np.array_equal(new_keep, keep): break
        keep = new_keep
    return keep, th, zr

K_CLIP = 4.0
keep, th_cut, zr = clip_fit(D_all, K_CLIP)
th_full, _, _ = maximize(lambda t: logpost1(t, D_all), start(D_all), H5)
D = sub(D_all, keep); N = len(D['y'])

print(f"removed {(~keep).sum()} of {len(keep)} SNe: index {np.where(~keep)[0]}, z-scores {np.round(zr[~keep], 1)}")
print("5 largest z of the SNe KEPT:", np.round(np.sort(zr[keep])[-5:], 2),
      "| 3 most negative z (not cut):", np.round(np.sort(zr)[:3], 2))
for Kc in (3, 3.5, 4, 5, 6): print(f"  K = {Kc}: removes {(~clip_fit(D_all, Kc)[0]).sum()}")
phys = lambda t: np.r_[t[:4], np.exp(t[4])]
print(pd.DataFrame({'no cut': phys(th_full), f'cut (K={K_CLIP})': phys(th_cut)},
                   index=['M [mag]', 'alpha [mag]', 'beta', 'gamma [mag]', 'sigma_int [mag]']).round(4))

**ANSWER:**

1. $\mathbf v^TC_i\mathbf v=\sigma_{m_B}^2+\alpha^2\sigma_{x_1}^2+\beta^2\sigma_c^2+2\alpha\,{\rm cov}(m_B,x_1)-2\beta\,{\rm cov}(m_B,c)-2\alpha\beta\,{\rm cov}(x_1,c)$. The off-diagonal terms are there because SALT2 fits on the light curve, so errors are correlated.
2. $\ln L=-\frac12\sum_i\left[\frac{r_i^2}{\sigma_i^2}+\ln(2\pi\sigma_i^2)\right]$, with $r_i=m_{B,i}-\mu(z_i)-M+\alpha x_{1,i}-\beta c_i-\gamma\Theta_i$ and $\sigma_i^2=\mathbf v^TC_i\mathbf v+\sigma_{\mu,{\rm pec},i}^2+\sigma_{\rm int}^2$.
3. $M\sim U(-21,-17)$ mag. SNe Ia are about $-19$ and the posterior is about 0.01 mag wide \
   $\alpha\sim U(-1,1)$ mag. The literature value is about 0.15, but the final value could be positive or negative \
   $\beta\sim U(0,8)$. The literature value is about 3, and redder means fainter \
   $\gamma\sim N(0,(0.1\ {\rm mag})^2)$. It is centered on the null with a width equal to the largest published steps of 0.05–0.1 mag. \
   $\ln\sigma_{\rm int}\sim U(\ln10^{-3},\ln1)$. It is scale-invariant from 1 mmag to 1 mag. 
4. I sampled $\ln\sigma_{\rm int}$ because it keeps $\sigma_{\rm int}>0$ automatically and the posterior is closer to Gaussian. That helps both with emcee and the Laplace evidence in 2d.
5. I removed them, using an iterative one-sided clip because the contaminants were made fainter. This removed 3 of 250 SNe (indices 11, 40, 224). The largest $z$ among the SNe kept is 3.24, as I tried not to cut the 3.2σ point. While this inflates error bars by only about 0.6%, the true concern is accidentally cutting real tail-data that should belong in this analysis. If I had not made these cuts, $\sigma_{\rm int}$ rises from 0.113 to 0.184 mag, and $\gamma$ moves from $-0.033$ to $-0.062$ mag, mimicking the published step. This makes all low-mass hosts articifically look fainter. 

### 2c - Sampling and convergence (12 pts)

1. Find the maximum a posteriori point. Then run **four independent chains** from starting points spread wider than the posterior (you choose how, and say how you chose). Any sampler is fine - your own Metropolis, `emcee` (one ensemble per chain), or `PyMC`.
2. Report $\hat R$ and the effective sample size for every parameter (`arviz.rhat`, `arviz.ess`, or `arviz.summary`), and say what values would have made you run longer.
3. Make a corner plot, and quote $\alpha$, $\beta$, $\gamma$ and $\sigma_{\rm int}$ as medians with 16th-84th percentile error bars.
4. Which parameters are correlated strongly? Comment on why. Then fix $M$ at its MAP value, refit $\gamma$, and compare that $\sigma_\gamma$ with the marginal one (cf. 1e).

`ArviZ` 1.x (install it with `conda install -c conda-forge pymc arviz`, as in `INSTALL.txt`) takes your chains as a dictionary of arrays shaped (chain, draw): `idata = az.from_dict({'posterior': {'alpha': a, 'beta': b}})`, then `az.summary(idata)`.

In [ ]:
# 2c: your work here

import emcee, corner, arviz as az

lp1 = lambda t: logpost1(t, D)
th_map, lp_max, cov_map = maximize(lp1, th_cut, H5)
sd_map = np.sqrt(np.diag(cov_map))
for n_, v_, s_, u_ in zip(['M', 'alpha', 'beta', 'gamma'], th_map[:4], sd_map[:4], ['mag', 'mag', '', 'mag']):
    print(f"MAP {n_:6s} = {fmt(v_, s_, u_)}")
print(f"MAP sigma_int = {np.exp(th_map[4]):.4f} mag;  max ln post = {lp_max:.2f}")

# four independent ensembles, started 5 sigma (marginal, from the MAP Hessian) from the MAP, in four different "corners"
nwalk, nsteps, ndim, BURN = 32, 8000, 5, 2000
signs = np.array([[1, 1, 1, 1, 1], [-1, -1, -1, -1, -1], [1, -1, 1, -1, 1], [-1, 1, -1, 1, -1]])
chains = []
for j in range(4):
    center = th_map + 5 * sd_map * signs[j]
    p0 = center + 0.1 * sd_map * np.random.default_rng(j).normal(size=(nwalk, ndim))
    assert all(np.isfinite(lp1(p)) for p in p0), f"chain {j}: start outside the prior"
    s = emcee.EnsembleSampler(nwalk, ndim, lp1)
    s.run_mcmc(p0, nsteps, progress=False)
    chains.append(s)
tau = np.array([c_.get_autocorr_time(quiet=True) for c_ in chains])
print("tau per chain (rows) and parameter (cols):\n", np.round(tau).astype(int))
THIN = max(1, int(tau.max() / 2))
print(f"burn-in {BURN}, thin {THIN}; mean acceptance {np.mean([c_.acceptance_fraction.mean() for c_ in chains]):.2f}")

# an "ensemble = one chain": walkers flattened step-major, shape (chain, draw)
post = np.stack([c_.get_chain(discard=BURN, thin=THIN).reshape(-1, ndim) for c_ in chains])
post[..., 4] = np.exp(post[..., 4])                         # sigma_int in mag
pnames = ['M', 'alpha', 'beta', 'gamma', 'sigma_int']
idata = az.from_dict({'posterior': {n_: post[..., i] for i, n_ in enumerate(pnames)}})
print(az.summary(idata))
print("independent ESS check from emcee tau:", np.round(4 * (nsteps - BURN) * nwalk / tau.max(axis=1).mean()), "total, (tau-based)")

# summary, corner, correlations
samples = post.reshape(-1, ndim)
for n_, u_, col_ in zip(pnames, ['mag', 'mag', '', 'mag', 'mag'], samples.T):
    lo_, md_, hi_ = np.percentile(col_, [16, 50, 84])
    print(f"{n_:10s} = {fmt_asym(md_, md_ - lo_, hi_ - md_, u_)}")
corner.corner(samples, labels=[r'$M$ [mag]', r'$\alpha$ [mag]', r'$\beta$', r'$\gamma$ [mag]', r'$\sigma_{\rm int}$ [mag]'],
              quantiles=[0.16, 0.5, 0.84], show_titles=True)
plt.show()
print(pd.DataFrame(np.corrcoef(samples.T), index=pnames, columns=pnames).round(2))
print(f"predicted rho(M, gamma) = -sqrt(N_massive/N) = {-np.sqrt(D['Th'].mean()):.2f}")

# fix M at its MAP value, refit the rest (as in 1e: fix one parameter, refit another)
idx = [1, 2, 3, 4]
x4, _, cov4 = maximize(lambda t4: logpost1(np.r_[th_map[0], t4], D), th_map[idx], H5[idx])
g_s = samples[:, 3]
sig_g_marg = 0.5 * (np.percentile(g_s, 84) - np.percentile(g_s, 16))
sig_g_fix = np.sqrt(cov4[2, 2])
rho_Mg = np.corrcoef(samples[:, 0], g_s)[0, 1]
print(f"sigma_gamma: marginal (MCMC) = {sig_g_marg:.4f} mag, marginal (Hessian) = {sd_map[3]:.4f} mag, M fixed = {sig_g_fix:.4f} mag")
print(f"ratio marginal/fixed = {sig_g_marg/sig_g_fix:.2f};  1/sqrt(1 - rho_Mg^2) = {1/np.sqrt(1 - rho_Mg**2):.2f}  (rho_Mg = {rho_Mg:.2f})")

**ANSWER:**

1. The MAP point is $M=-19.275\pm0.014$ mag, $\alpha=0.1451\pm0.0096$ mag, $\beta=2.75\pm0.10$, $\gamma=-0.033\pm0.019$ mag and $\sigma_{\rm int}=0.1131$ mag. I ran four emcee ensembles of 32 walkers for 8000 steps, with 2000 burn-in. Each was started $5\sigma$ from the MAP on every axis, in four different corners. That is well outside the posterior, so agreement would be a real test.
2. $\hat R=1.00$ for all five parameters. ArviZ gives ESS bulk of $2.2$–$2.3\times10^4$ and ESS tail of about $2.2\times10^4$. A $\tau$-based check gives $1.2\times10^4$. ArviZ’s number is probably optimistic because I flatten 32 walkers per chain, so I trust the $\tau$-based one. I would have run longer if any $\hat R>1.01$, or any ESS (bulk or tail) fell below about 400.
3. 16th–84th percentiles: \
$\alpha=0.1452^{+0.009}_{-0.009}$ mag \
$\beta=2.75\pm0.1$ \
$\gamma=-0.032\pm0.02$ mag \
$\sigma_{\rm int}=0.1144^{+0.0089}_{-0.0084}$ mag \
4. The strongest correlation is $M$–$\gamma$. $M$ is effectively the mean magnitude of the low-mass hosts and $M+\gamma$ that of the massive ones, so raising $M$ forces $\gamma$ down. Fixing $M$ would understate the step’s uncertainty. 

### 2d - Model comparison (10 pts)

Compare $M_0$ ($\gamma = 0$) with $M_1$ ($\gamma$ free).

1. Compute $2\ln B_{1,0}$ with the Laplace approximation (Day 13), with a prior on $\gamma$ you state and defend. How does it change if that prior is twice as wide?
2. Compute $\textrm{BIC}_0 - \textrm{BIC}_1$ for comparison.
3. How strong is your answer on Day 8's scale (0-2 weak, 2-6 positive, 6-10 strong, above 10 very strong)? Compare $\gamma$ with the published $-0.06$ mag. Could your 250 supernovae have detected a step that size? (One way: your $\sigma_\gamma$ tells you how many $\sigma$ a $-0.06$ step would be.)

In [ ]:
# 2d: your work here

def laplace_lnZ(logf, th0, h):
    """ln Z ~ ln[L P](peak) + (d/2) ln 2pi + 1/2 ln|C|, with C the inverse Hessian of -ln(L P) (Day 13)."""
    x, lmax, cov = maximize(logf, th0, h)
    return lmax + 0.5 * len(x) * np.log(2 * np.pi) + 0.5 * np.linalg.slogdet(cov)[1], x

i4 = [0, 1, 2, 4]                                              # M0 drops gamma
lnZ0, th0_hat = laplace_lnZ(lambda t: logpost0(t, D), th_map[i4], H5[i4])
lnZ1, th1_hat = laplace_lnZ(lambda t: logpost1(t, D, GSD), th_map, H5)
lnZ1w, _      = laplace_lnZ(lambda t: logpost1(t, D, 2 * GSD), th_map, H5)
print(f"ln Z0 = {lnZ0:.2f}, ln Z1 = {lnZ1:.2f} (prior sd {GSD}), ln Z1 = {lnZ1w:.2f} (prior sd {2*GSD})")
print(f"2 ln B10 = {2*(lnZ1 - lnZ0):+.2f}   (gamma prior sd {GSD} mag)")
print(f"2 ln B10 = {2*(lnZ1w - lnZ0):+.2f}   (gamma prior sd {2*GSD} mag)")

kde0 = stats.gaussian_kde(g_s)(0.0)[0]
print(f"Savage-Dickey 2 ln B10 = {-2*(np.log(kde0) - stats.norm.logpdf(0, 0, GSD)):+.2f}")

# BIC 
_, ll1, _ = maximize(lambda t: loglike(t, D), th_map, H5)
_, ll0, _ = maximize(lambda t: loglike([t[0], t[1], t[2], 0.0, t[3]], D), th_map[i4], H5[i4])
BIC1, BIC0 = -2*ll1 + 5*np.log(N), -2*ll0 + 4*np.log(N)
print(f"ln Lmax: M0 = {ll0:.2f}, M1 = {ll1:.2f};  BIC0 = {BIC0:.2f}, BIC1 = {BIC1:.2f};  BIC0 - BIC1 = {BIC0 - BIC1:+.2f}")

g_hat = np.median(g_s)
print(f"gamma = {fmt(g_hat, sig_g_marg, 'mag')};  distance from 0: {abs(g_hat)/sig_g_marg:.2f} sigma;  "
      f"from -0.06: {(g_hat + 0.06)/sig_g_marg:+.2f} sigma")
print(f"a -0.06 mag step is {0.06/sig_g_marg:.2f} sigma_gamma;  3-sigma detection needs sigma_gamma = 0.020 mag "
      f"-> {(sig_g_marg/0.02)**2:.1f}x as many SNe")

**ANSWER:**

1. I used $\gamma\sim N(0,(0.1\ {\rm mag})^2)$. The Laplace evidence gives $2\ln B_{1,0}=-0.41$, and the MCMC samples agrees ($-0.37$). With the prior twice as wide, $2\ln B_{1,0}=-1.69$. The values are negative because $\hat\gamma$ is only $1.7\sigma$ from zero, which is not enough to compensate for the extra parameter.
2. $\mathrm{BIC}_0-\mathrm{BIC}_1=-2.49$. It is more negative than the Laplace values because BIC implicitly uses a wider prior on $\gamma$.
3. All three numbers are weak to at most positive evidence for no step, and none is evidence against one. $\gamma=-0.032\pm0.019$ mag is $1.7\sigma$ from zero and $1.5\sigma$ from the published $-0.06$ mag, so it is consistent with both. On the scale given, I would classify this in the 0-2 scale. 

### 2e - Testing your pipeline (8 pts)

Design one test of your pipeline that would come out differently if your pipeline were wrong, and run it. State **before** you run it what result would count as a failure. One suggestion: generate fake supernovae at your redshifts, stretches and colors from parameters you choose (including a $\gamma$ you choose), fit 30 or more such datasets (maximum-likelihood fits with curvature error bars are fine here; you don't need MCMC for each), and check how often your 68% intervals contain the truth.

I simulate 300 fake datasets from a truth I choose ($M=-19.3$ mag, $\alpha=0.15$ mag, $\beta=3.0$, $\gamma=-0.06$ mag, $\sigma_{\rm int}=0.10$ mag), using my real redshifts, stretches, colors, host masses and covariances. I refit each one and compute pulls $(\hat\theta-\theta_{\rm true})/\sigma_\theta$. The pipeline fails if, for any parameter, the 68% coverage is more than 3 binomial errors from 68.3%, the mean pull is above 0.3 in magnitude, or the std of the pulls is outside $1\pm0.15$.

In [ ]:
# 2e: your work here

truth = np.array([-19.3, 0.15, 3.0, -0.06, np.log(0.10)])
N_SIM, N_SIM2 = 300, 100

Cm = np.empty((N, 3, 3))
Cm[:, 0, 0], Cm[:, 1, 1], Cm[:, 2, 2] = D['vmB'], D['vx1'], D['vc']
Cm[:, 0, 1] = Cm[:, 1, 0] = D['cmx']; Cm[:, 0, 2] = Cm[:, 2, 0] = D['cmc']; Cm[:, 1, 2] = Cm[:, 2, 1] = D['cxc']
Lc = np.linalg.cholesky(Cm)

def simulate(th_t, rng, n_contam=0, shift=1.2):
    M, a, b, g, lns = th_t
    nz = np.einsum('nij,nj->ni', Lc, rng.normal(size=(N, 3))) 
    e = nz[:, 0] + a * nz[:, 1] - b * nz[:, 2] 
    y = (M - a*D['x1'] + b*D['col'] + g*D['Th'] + e
         + rng.normal(0, np.sqrt(D['pec2'] + np.exp(2 * lns)))) 
    if n_contam: y[rng.choice(N, n_contam, replace=False)] += shift
    return dict(D, y=y)

def ml_pipeline(Ds):
    x, _, cov = maximize(lambda t: loglike(t, Ds), start(Ds), H5)
    return x, np.sqrt(np.diag(cov))

def clip_pipeline(Ds):
    kp, _, _ = clip_fit(Ds, K_CLIP)
    return ml_pipeline(sub(Ds, kp))

def run(pipeline, n, seed, **kw):
    rng = np.random.default_rng(seed); est, err = np.empty((n, 5)), np.empty((n, 5))
    with np.errstate(all='ignore'):
        for i in range(n): est[i], err[i] = pipeline(simulate(truth, rng, **kw))
    return est, err

def report(est, err, label):
    pull = (est - truth) / err; ok = np.isfinite(pull).all(1); p = pull[ok]; n = ok.sum()
    cov_, mean_, sd_ = (np.abs(p) < 1).mean(0), p.mean(0), p.std(0, ddof=1)
    tolc = 3 * np.sqrt(0.683 * 0.317 / n)
    verdict = np.where((np.abs(cov_ - 0.683) < tolc) & (np.abs(mean_) < 0.3) & (np.abs(sd_ - 1) < 0.15), 'pass', 'FAIL')
    print(f"\n{label}: {n}/{len(est)} usable fits, coverage tolerance ±{tolc:.3f}")
    print(pd.DataFrame({'mean pull': mean_, 'std pull': sd_, 'coverage': cov_, 'verdict': verdict}, index=names5).round(3))

report(*run(ml_pipeline, N_SIM, 1), "A. clean fakes, ML fit")
report(*run(ml_pipeline, N_SIM2, 2, n_contam=3), "B. 3 contaminants injected, NO clip")
report(*run(clip_pipeline, N_SIM2, 3, n_contam=3), "C. 3 contaminants injected, with the clip")

**ANSWER:**

Coverages are 63–70% and pull stds are 0.92–1.12. The one near-miss is the $\ln\sigma_{\rm int}$ mean pull of $-0.28$, which is just inside my limit. With 3 contaminants and no clip, the test fails: the $M$ pull is $+0.84$ and the $\ln\sigma_{\rm int}$ pull is $+7.9$, which makes sense since the non-clipped data should not pass. With the clip, every parameter passes again.

## Part 3: The referee report (25 pts)

White dwarfs are the cores of dead stars like the Sun. Your file `p3_wd_masses.csv` holds 600 real DA (hydrogen-atmosphere) white dwarfs from the 100 pc sample of [Kilic et al. (2020)](https://arxiv.org/abs/2006.00323), ApJ 898, 84: `name`, `teff_K`, `mass_Msun`, `mass_err_Msun`. Most single white dwarfs have masses near $0.6\,M_\odot$; Kilic et al. report a narrow peak there plus an excess of more massive ones, around $0.7$-$0.9\,M_\odot$, which merger products alone do not explain.

The notebook `labs/midterm/ai_solution.ipynb` fits your masses with a two-population model using `emcee`. Run it from that directory, after setting `NETID` in its first cell. It runs top to bottom and the plots look nice, but it's wrong in at least three substantive ways - and if you don't look at your data, you'll fool yourself the same way.

### 3a - The report (20 pts)

1. **Identify** at least three substantive errors (not style; "it used 32 walkers and I'd use 48" is not one).
2. **Demonstrate** each on *your* file: show, with a number or a plot, what the error does to the answer. A demonstration can come out negative: if an error doesn't move your numbers, show that, and say what it would take for it to matter.
3. **State** the correct treatment, and run it where you can.

In [ ]:
# 3a: your work here
import emcee, arviz as az

wd = load('p3_wd_masses')
mass, merr = wd['mass_Msun'].to_numpy(), wd['mass_err_Msun'].to_numpy()
zero = np.zeros_like(mass)
pn = ['mu1', 'sigma1', 'mu2', 'sigma2', 'f']

med, sd = np.median(mass), np.std(mass)
clip_keep = np.abs(mass - med) < 3 * sd                             # the AI's cut, exactly
print(f"np.std = {sd:.3f} Msun vs robust (MAD) sigma = {1.4826 * np.median(np.abs(mass - med)):.3f} Msun")
print(f"AI's window {med - 3*sd:.2f}-{med + 3*sd:.2f} Msun removes {(~clip_keep).sum()} stars: {np.sort(mass[~clip_keep]).round(2)}")
print(f"errors: median {np.median(merr):.4f}, 16-84% {np.percentile(merr, [16, 84]).round(4)}, 99% {np.percentile(merr, 99):.3f} Msun")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].hist(mass, bins=60, color='0.8', edgecolor='k', lw=0.3); ax[0].set_yscale('log')
ax[0].axvspan(med - 3*sd, med + 3*sd, color='C0', alpha=0.12)
ax[0].set_xlabel('mass [Msun]'); ax[0].set_ylabel('stars per bin'); ax[0].set_title("AI's 3σ window (blue)")
ax[1].plot(mass, merr, 'k.', ms=3); ax[1].plot(mass[~clip_keep], merr[~clip_keep], 'ro', mfc='none')
ax[1].set_xlabel('mass [Msun]'); ax[1].set_ylabel('mass error [Msun]'); ax[1].set_title('per-star errors (clipped stars circled)')
plt.tight_layout(); plt.show()

def log_like(p, m, e): 
    mu1, s1, mu2, s2, f = p
    v1, v2 = s1**2 + e**2, s2**2 + e**2
    l1 = np.log(f)    - 0.5 * ((m - mu1)**2 / v1 + np.log(2 * np.pi * v1))
    l2 = np.log1p(-f) - 0.5 * ((m - mu2)**2 / v2 + np.log(2 * np.pi * v2))
    return np.sum(np.logaddexp(l1, l2))

def log_prob(p, m, e): 
    mu1, s1, mu2, s2, f = p
    if not (0.3 < mu1 < 1.3 and 0.3 < mu2 < 1.3 and 0.005 < s1 < s2 < 0.5 and 0 < f < 1): return -np.inf
    return log_like(p, m, e)

def ai_log_prob(p, m): 
    mu1, s1, mu2, s2, f = p
    if not (0.3 < mu1 < 1.3 and 0.3 < mu2 < 1.3 and 0.005 < s1 < 0.5 and 0.005 < s2 < 0.5 and 0 < f < 1): return -np.inf
    n = lambda mu, s: np.exp(-0.5 * ((m - mu) / s)**2) / (s * np.sqrt(2 * np.pi))
    return np.sum(np.log(f * n(mu1, s1) + (1 - f) * n(mu2, s2)))

def nll(p, m, e):
    v = log_prob(p, m, e); return -v if np.isfinite(v) else 1e30

def ml_fit(m, e, n_start=40, seed=0):  
    rng_ = np.random.default_rng(seed); best = None
    for _ in range(n_start):
        p0 = np.array([rng_.uniform(.45, .70), rng_.uniform(.02, .15), rng_.uniform(.65, 1.10), rng_.uniform(.03, .30), rng_.uniform(.3, .95)])
        if p0[1] >= p0[3]: continue
        r = minimize(nll, p0, args=(m, e), method='Nelder-Mead',
                     options=dict(xatol=1e-7, fatol=1e-9, maxiter=4000, maxfev=4000, adaptive=True))
        if best is None or r.fun < best.fun: best = r
    return best.x, -best.fun

def disp_centers(rng_, n=4): 
    out = []
    while len(out) < n:
        c_ = np.array([rng_.uniform(.45, .70), rng_.uniform(.02, .20), rng_.uniform(.65, 1.10), rng_.uniform(.03, .30), rng_.uniform(.2, .95)])
        if c_[1] < c_[3]: out.append(c_)
    return out

DE = [(emcee.moves.DEMove(), 0.8), (emcee.moves.DESnookerMove(), 0.2)]   # differential-evolution moves help hop between modes

def sample(logp, args, centers, nsteps=6000, nwalk=32, seed=0, moves=None):    # one emcee ensemble per chain
    rng_ = np.random.default_rng(seed); out = []
    for c_ in centers:
        while True:
            p0 = c_ * (1 + 0.02 * rng_.normal(size=(nwalk, len(c_))))
            if all(np.isfinite(logp(p, *args)) for p in p0): break
        s = emcee.EnsembleSampler(nwalk, len(c_), logp, args=args, moves=moves)
        s.run_mcmc(p0, nsteps, progress=False); out.append(s)
    return out

def diagnose(chains, burn, label, use=None):
    chains = [chains[i] for i in (range(len(chains)) if use is None else use)]
    tau = np.array([s.get_autocorr_time(tol=0) for s in chains]); thin = max(1, int(tau.max() / 2))
    post = np.stack([s.get_chain(discard=burn, thin=thin).reshape(-1, 5) for s in chains])      # (chain, draw, param)
    idata = az.from_dict({'posterior': {n_: post[..., i] for i, n_ in enumerate(pn)}})
    print(f"\n{label}\n  tau max = {tau.max():.0f} steps; (steps kept)/tau = {(chains[0].iteration - burn) / tau.max():.0f}")
    print(az.summary(idata)[['r_hat', 'ess_bulk', 'ess_tail']])
    print("  per-chain medians (rows = chains):\n", np.round(np.median(post, axis=1), 3))
    return post

BURN = 5000
chains_c = sample(log_prob, (mass, merr), disp_centers(np.random.default_rng(11)), nsteps=20000, seed=11, moves=DE)
lpmax = np.array([s.get_log_prob(discard=BURN).max() for s in chains_c])
use = [i for i in range(4) if lpmax[i] > lpmax.max() - 5]           # chains whose peak ln posterior is within 5 of the best
print("peak ln posterior per chain:", lpmax.round(1), "-> using chains", use)
post_c = diagnose(chains_c, BURN, f"CORRECTED pipeline (chains {use})", use=use)
samples_c = post_c.reshape(-1, 5); sd_post = samples_c.std(0)
for n_, u_, col_ in zip(pn, ['Msun'] * 4 + [''], samples_c.T):
    lo_, md_, hi_ = np.percentile(col_, [16, 50, 84]); print(f"{n_:7s} = {fmt_asym(md_, md_ - lo_, hi_ - md_, u_)}")
th_c = np.median(samples_c, axis=0); xg = np.linspace(0.2, mass.max(), 400)
g_ = lambda mu, s: stats.norm.pdf(xg, mu, s)
plt.hist(mass, 60, density=True, color='0.8'); plt.yscale('log'); plt.ylim(1e-3, None)
plt.plot(xg, th_c[4] * g_(th_c[0], th_c[1]) + (1 - th_c[4]) * g_(th_c[2], th_c[3]), 'k', label='posterior-median mixture (intrinsic)')
plt.xlabel('mass [Msun]'); plt.legend(); plt.show()

#  Errors 1 and 2
rows = {}
for clip in (True, False):
    for use_err in (False, True):
        k_ = clip_keep if clip else np.ones_like(clip_keep)
        th, _ = ml_fit(mass[k_], (merr if use_err else zero)[k_])
        rows[f"clip {'on' if clip else 'off'}, errors {'used' if use_err else 'ignored'}"] = np.r_[th, k_.sum()]
tab = pd.DataFrame(rows, index=pn + ['N']).T
print(tab.round(4))
print("\nshift from the corrected result (clip off, errors used), in units of the corrected posterior sigma:")
print(((tab[pn] - tab.loc['clip off, errors used', pn].to_numpy(float)) / sd_post).round(2))
print(f"\nmedian error {np.median(merr):.4f} Msun vs fitted sigma1 = {th_c[1]:.4f} Msun; "
      f"fraction of stars with error > sigma1/2: {np.mean(merr > th_c[1] / 2):.2f}   (AI claimed 'about 0.01')")

#  Error 3: 
def gelman_rubin(x_): 
    n_, _ = x_.shape; B = n_ * np.var(x_.mean(axis=0), ddof=1); W = np.mean(np.var(x_, axis=0, ddof=1))
    return np.sqrt(((n_ - 1) / n_ * W + B / n_) / W)

mc = mass[clip_keep]; lo7, hi7 = mc[mc < 0.7], mc[mc >= 0.7]
p0_ai = np.array([lo7.mean(), lo7.std(), hi7.mean(), hi7.std(), lo7.size / mc.size])
np.random.seed(42)
s_ai = emcee.EnsembleSampler(32, 5, ai_log_prob, args=(mc,))
s_ai.run_mcmc(p0_ai + 1e-3 * np.random.randn(32, 5), 3000, progress=False)
ch_ai = s_ai.get_chain(discard=500); tau_ai = s_ai.get_autocorr_time(tol=0)
print("AI's own R-hat (walkers of ONE ensemble, 1e-3 ball):", {n_: round(float(gelman_rubin(ch_ai[:, :, i])), 3) for i, n_ in enumerate(pn)})
print("AI's tau:", np.round(tau_ai).astype(int), f"-> 2500 steps / tau_max = {2500 / tau_ai.max():.0f}")
chains_ai = sample(ai_log_prob, (mc,), disp_centers(np.random.default_rng(3)), seed=3)
diagnose(chains_ai, 2000, "AI's model and data (clipped, no errors), but 4 OVERDISPERSED ensembles")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot(s_ai.get_chain()[:, :, 2], lw=0.3, alpha=0.5); ax[0].set_title("AI: 32 walkers started within 1e-3 of one point")
for j, s_ in enumerate(chains_ai): ax[1].plot(s_.get_chain()[:, :, 2].mean(axis=1), lw=0.8, label=f'ensemble {j}')
ax[1].set_title('4 overdispersed ensembles (walker mean)'); ax[1].legend(fontsize=7)
for a in ax: a.set_xlabel('step'); a.set_ylabel('mu2 [Msun]')
plt.tight_layout(); plt.show()

**REFEREE REPORT:**

1. The 3σ clip deletes the signal. The AI clips about the median using np.std of the whole sample, and the tail inflates that σ. The correct treatment is no clip, since the masses are physical measurements and the tail is the signal. If real bad fits existed, they should be modelled or flagged externally.
2. The per-star mass errors are ignored on an unchecked claim. The AI asserts the median error is "about 0.01 M☉, far smaller than the population widths." The correct treatment replaces $\sigma_k^2$ by $\sigma_k^2+e_i^2$ for each star.
3. The convergence check cannot fail. $\hat R$ is computed across the walkers of one ensemble, all started within 1e-3 of the same point. They are not independent chains, so $\hat R\approx1$ by construction. There is no $\tau$ or ESS, and no test for label switching. 

### 3b - AI-use appendix (5 pts)

1. Which tools did you use, for what, what did they get wrong, and how did you catch it? "I used ChatGPT to help" is useless; "none" is fine if that's true. I promise I don't care which - just be honest about what you did.
2. What would have gone wrong in your answer if you had not checked the AI's output? Name the specific thing, in two or three sentences.

**AI-USE APPENDIX:**

1. As with previous labs, I used Claude's Sonnet 5.5 AI to generate some boilerplate code for this lab, particularly for the more mathematical helper functions where there were additional parameters and syntax I wanted to check on. Claude also contributed to make the graphs cleaner for the data they could show and helped format the fstring outputs to be more readable. For this lab specifically, I also used Claude to help clean-up some responses involving heavy mathematical notation (Question 2, part 3), since my non-LaTeX solutions were incredibly difficult to read initially. This means that Claude had more influence in the final submission of this midterm than it previously had on prior labs. The responses and analysis I had for each question were my own.
If you notice that this appendix is very similar to the ones submitted on the previous labs, that is a choice made intentionally, since AI was used the same way on both labs.
2. For Question 2A, part 4: it was good that I checked on the contaminants independently. The original iterations of the graphs I had didn't highlight any specific data points as part of the graph, but the AI-generated ones suddenly had 4 points circled in different positions on the first 3 graphs. It was worth my time to understand how and why this happened, since Claude could have just as easily circled any points and claimed they didn't fit correctly. 

Get started. You know how to do all of this. If not, there's something in a lab or class notebook that can help. Slack me with questions. Best of luck.